In [1]:
import os                                                      
import logging                                           
from typing import TypedDict                                  
from dotenv import load_dotenv                                 

os.environ['ANONYMIZED_TELEMETRY'] = 'False'             
logging.getLogger('httpx').setLevel(logging.WARNING)            

from langchain_openai import ChatOpenAI, OpenAIEmbeddings  
from langchain_chroma import Chroma                           
from langchain_core.prompts import ChatPromptTemplate     
from langchain_core.output_parsers import StrOutputParser  
from typing import TypedDict, List, Optional
from langchain_core.documents import Document
from langgraph.graph import StateGraph, START, END

load_dotenv()                                                  
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)   

print('Imports ready')
print(f'   Model: {llm.model_name}')

Imports ready
   Model: gpt-4o-mini


Load the Vector Store

In [2]:
persist_dir = r'C:\Users\USER\rag_course\chroma_db_domain'

# Convert data to vector
embeddings = OpenAIEmbeddings()

# stores vectors on disk and search
vectorstore = Chroma(
    persist_directory=persist_dir,
    embedding_function=embeddings
)

# Retriever — returns top 4 similar chunks
retriever = vectorstore.as_retriever(search_kwargs={'k': 4})

print(f'   Chunks in store: {vectorstore._collection.count()}')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


   Chunks in store: 80


Relevance Grader Chain

In [3]:
grader_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a relevance grader. Given a question and a document, decide if the document is relevant. Answer with exactly one word: YES or NO.'),
    ('human', 'Query: {query}\n\nDocument:\n{document}'),
])

relevance_grader = grader_prompt | llm | StrOutputParser()

print('Relevance chain ready')

Relevance chain ready


Test the Grader

In [4]:
result = relevance_grader.invoke({
    'query': 'How do I treat cassava mosaic disease?',
    'document': 'Cassava Mosaic Disease is viral. Control: disease-free cuttings, resistant varieties.',
}).strip().upper()

print('Result:', result)

Result: YES


In [5]:
result = relevance_grader.invoke({
    'query': 'How do I treat cassava mosaic disease?',
    'document': 'Rice Blast is a fungal disease of rice.',
}).strip().upper()

print('Result:', result)

Result: NO


Query Rewriter Chain

In [6]:
rewriter_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You rewrite queries to improve document retrieval. Output only the rewritten query, nothing else.'),
    ('human', '{query}'),
])

query_rewriter = rewriter_prompt | llm | StrOutputParser()

print('query_rewriter ready')

query_rewriter ready


Test the Rewriter

In [7]:
original = 'How do I treat tomato blight?'
rewritten = query_rewriter.invoke({'query': original}).strip()

print('Original: ', original)
print('Rewritten:', rewritten)

Original:  How do I treat tomato blight?
Rewritten: What are effective treatments for tomato blight?


The LangGraph State

In [8]:
class CRAGState(TypedDict):
    '''
    State for Corrective RAG — flows through every node in the graph.
    '''
    
    query: str          #  Original user question
    rewritten_query: Optional[str]        # Rewritten query
    retrieved_docs: List[Document]    # All retrieved chunks
    corrected_docs: List[Document]     # Only the relevant chunks
    retry_count: int
    answer: str                 # The final answer    
    
print(f'   Fields: {list(CRAGState.__annotations__.keys())}')

   Fields: ['query', 'rewritten_query', 'retrieved_docs', 'corrected_docs', 'retry_count', 'answer']


The retrieve_node Function

In [9]:
def retrieve_node(state: CRAGState) -> dict:
    '''
    Retrieve documents for the current question (original or rewritten)
    '''
    
    query = state.get('rewritten_query') or state['query']
    
    docs = retriever.invoke(query)
    print(f'Retrieved {len(docs)} docs for: "{query}"')
    
    return {
        'retrieved_docs': docs
    }


Test the Node Directly

In [10]:
test_state = {'query': 'How do I treat cassava mosaic disease?'}

result = retrieve_node(test_state)
print(f'\nReturned {len(result["retrieved_docs"])} docs')
print(f'First doc preview: {result["retrieved_docs"][0].page_content}')

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Retrieved 4 docs for: "How do I treat cassava mosaic disease?"

Returned 4 docs
First doc preview: Management
O Use of disease free planting materials.
O The uprooted plants should be carried 
away from the field and exposed to the 
sunlight for drying and then burned to kill 
the viruses.
O Do not plant cassava close to infected 
fields as the insect vectors will transport 
the disease to healthy plants.


Filter relevant node

In [11]:
def filter_relevant_node(state: CRAGState) -> dict:
    """Grade each retrieved doc — keep only the relevant ones."""
    relevant_docs = []
    for doc in state['retrieved_docs']:
        relevance = relevance_grader.invoke({
            'query': state['query'],
            'document': doc.page_content[:500],
        }).strip().upper()
        if 'YES' in relevance:
            relevant_docs.append(doc)

    print(f'Graded: {len(relevant_docs)}/{len(state["retrieved_docs"])} relevant')
    return {'corrected_docs': relevant_docs}


print('filter_relevant_node ready')

filter_relevant_node ready


Test

In [12]:
state = {'query': 'How do I treat cassava mosaic disease?'}

state.update(retrieve_node(state))
state.update(filter_relevant_node(state))

print(f'\nCorrected docs: {len(state["corrected_docs"])}')

for i, d in enumerate(state['corrected_docs']):
    print(f'[{i}] {d.page_content}')
    print()

Retrieved 4 docs for: "How do I treat cassava mosaic disease?"
Graded: 4/4 relevant

Corrected docs: 4
[0] Management
O Use of disease free planting materials.
O The uprooted plants should be carried 
away from the field and exposed to the 
sunlight for drying and then burned to kill 
the viruses.
O Do not plant cassava close to infected 
fields as the insect vectors will transport 
the disease to healthy plants.

[1] Cassava
O Disease: Cassava Mosaic Virus (CMV) 
disease 
O Causative organism: Trasmitted by 
white fly (Bemisia tabaci)

[2] Common Crop Diseases and Their Control

1. Cassava Mosaic Disease
Affected Crop: Cassava
Symptoms: Yellowing and mottling of leaves, stunted growth, reduced yield.
Control/Cure: Use disease-free cuttings, plant resistant varieties, and remove infected plants early.


2. Maize Smut
Affected Crop: Maize
Symptoms: Large grey or black galls on ears, stalks, and leaves.
Control/Cure: Remove and destroy infected plants, rotate crops, and plant resistant h

The rewrite_question_node function

In [13]:
def rewrite_question_node(state: CRAGState) -> dict:
    """Rewrite the query to improve retrieval (used on retry)."""
    new_query = query_rewriter.invoke({'query': state['query']}).strip()
    print(f'Rewritten: "{new_query}"')
    return {
        'rewritten_query': new_query,
        'retry_count': (state.get('retry_count') or 0) + 1,
    }


print('rewrite_question_node ready')

rewrite_question_node ready


In [14]:
state = {'query': 'How do I treat cassava mosaic?', 'retry_count': 1}
result = rewrite_question_node(state)
print(f'\nNew retry_count: {result["retry_count"]}')     

Rewritten: "What are the treatment options for cassava mosaic disease?"

New retry_count: 2


The answer_from_docs_node Function

In [15]:
def answer_from_docs_node(state: CRAGState) -> dict:
    '''Answer the query using the corrected (relevant) docs.'''
    
    context = '\n\n'.join(doc.page_content for doc in state['corrected_docs'])

    answer_prompt = ChatPromptTemplate.from_messages([
        ('system', 'Answer using ONLY the context below. If the answer is not in the context, say "I don\'t know".'),
        ('human', 'Context:\n{context}\n\nQuery: {query}'),
    ])

    answer_chain = answer_prompt | llm | StrOutputParser()
    answer = answer_chain.invoke({'context': context, 'query': state['query']})

    print('Answered from corrected docs')
    return {'answer': answer}


print('answer_from_docs_node ready')

answer_from_docs_node ready


Test

In [16]:
state = {'query': 'How do I treat cassava mosaic disease?'}
state.update(retrieve_node(state))
state.update(filter_relevant_node(state))
state.update(answer_from_docs_node(state))

print(f'\nAnswer:\n{state["answer"]}')

Retrieved 4 docs for: "How do I treat cassava mosaic disease?"
Graded: 4/4 relevant
Answered from corrected docs

Answer:
To treat cassava mosaic disease, use disease-free cuttings, plant resistant varieties, and remove infected plants early.


The answer_directly_node Function (Fallback)

In [17]:
def answer_directly_node(state: CRAGState) -> dict:
    """Fall back — answer directly from the LLM when retrieval can't help."""
    direct_prompt = ChatPromptTemplate.from_messages([
        ('system', 'Answer the query directly using your own knowledge. Be honest if you don\'t know.'),
        ('human', '{query}'),
    ])

    direct_chain = direct_prompt | llm | StrOutputParser()
    answer = direct_chain.invoke({'query': state['query']})

    print('Fallback: answered directly')
    return {'answer': answer}


print('answer_directly_node ready')

answer_directly_node ready


In [18]:
state = {'query': 'What is the capital of France?'}
state.update(answer_directly_node(state))

print(f'\nFallback answer:\n{state["answer"]}')

Fallback: answered directly

Fallback answer:
The capital of France is Paris.


The Router Function

In [19]:
def route_after_grading(state: CRAGState) -> str:
    """Decide next step: answer, retry, or fallback."""
    
    # Count relevant docs
    n_relevant = len(state.get('corrected_docs', []))
    
    # True if we already retried once
    retried = state.get('retry_count', 0) > 0

    if n_relevant >= 1:
        print(f'Route: {n_relevant} relevant docs → answer')
        return 'answer'
    elif not retried:
        print(f'Route: no relevant docs → retry')
        return 'retry'
    else:
        print(f'Route: retry failed → fallback')
        return 'fallback'


print('route_after_grading ready')

route_after_grading ready


 Test All Three Branches

In [20]:
# Branch 1 — has relevant docs
print('--- Branch 1 ---')
route_after_grading({'corrected_docs': ['doc1', 'doc2'], 'retry_count': 0})

# Branch 2 — no docs, no retry yet
print('\n--- Branch 2 ---')
route_after_grading({'corrected_docs': [], 'retry_count': 0})

# Branch 3 — no docs, already retried
print('\n--- Branch 3 ---')
route_after_grading({'corrected_docs': [], 'retry_count': 1})

--- Branch 1 ---
Route: 2 relevant docs → answer

--- Branch 2 ---
Route: no relevant docs → retry

--- Branch 3 ---
Route: retry failed → fallback


'fallback'

The Graph Assembly

In [21]:
builder = StateGraph(CRAGState)

# Register all nodes
builder.add_node('retrieve', retrieve_node)
builder.add_node('filter_relevant', filter_relevant_node)
builder.add_node('rewrite_question', rewrite_question_node)
builder.add_node('answer_from_docs', answer_from_docs_node)
builder.add_node('answer_directly', answer_directly_node)

# Linear edges: START → retrieve → filter
builder.add_edge(START, 'retrieve')
builder.add_edge('retrieve', 'filter_relevant')

# Conditional routing after grading
builder.add_conditional_edges(
    'filter_relevant',
    route_after_grading,
    {
        'answer': 'answer_from_docs',
        'retry': 'rewrite_question',
        'fallback': 'answer_directly',
    }
)

# Retry loop and exits
builder.add_edge('rewrite_question', 'retrieve')
builder.add_edge('answer_from_docs', END)
builder.add_edge('answer_directly', END)

crag_app = builder.compile()

print('Corrective RAG graph compiled')

Corrective RAG graph compiled


The Live Test

In [22]:
result = crag_app.invoke({'query': 'How do I treat cassava mosaic disease?'})
print(f'\nFinal answer:\n{result["answer"]}')

Retrieved 4 docs for: "How do I treat cassava mosaic disease?"
Graded: 4/4 relevant
Route: 4 relevant docs → answer
Answered from corrected docs

Final answer:
To treat cassava mosaic disease, use disease-free cuttings, plant resistant varieties, and remove infected plants early.


Test 2 — The Retry Path

In [23]:
result = crag_app.invoke({'query': 'How do I treat tomato blight?'})
print(f'\nFinal answer:\n{result["answer"]}')

Retrieved 4 docs for: "How do I treat tomato blight?"
Graded: 2/4 relevant
Route: 2 relevant docs → answer
Answered from corrected docs

Final answer:
I don't know.


Test 3 — General Knowledge

In [24]:
result = crag_app.invoke({'query': 'What is the capital of France?'})
print(f'\nFinal answer:\n{result["answer"]}')

Retrieved 4 docs for: "What is the capital of France?"
Graded: 0/4 relevant
Route: no relevant docs → retry
Rewritten: "What is the name of the capital city of France?"
Retrieved 4 docs for: "What is the name of the capital city of France?"
Graded: 0/4 relevant
Route: retry failed → fallback
Fallback: answered directly

Final answer:
The capital of France is Paris.
